# CS589 Homework 2 — ElasticSearch 7.9.0 (Instructor-Aligned Version)
This notebook strictly follows the assignment instructions:
- ES + Kibana 7.9.0
- Scripted TF‑IDF similarity
- Default BM25
- Default LMDirichlet similarity
- Analyzer: whitespace + lowercase + porter_stem
- Report 9 NDCG@10 scores

⚠️ Make sure ES 7.9.0 is running before executing.

In [ ]:
!pip -q install elasticsearch==7.9.1

## Configuration

In [ ]:

from pathlib import Path
from elasticsearch import Elasticsearch
from elasticsearch.helpers import streaming_bulk
import json, re

ES_HOST = "http://localhost:9200"
DATA_DIR = Path("PUT_YOUR_DATASET_FOLDER_HERE")

LANGS = ["python", "java", "javascript"]

QID2ALL = {l: DATA_DIR / f"{l}_qid2all.txt" for l in LANGS}
COSIDF  = {l: DATA_DIR / f"{l}_cosidf.txt" for l in LANGS}

def index_name(lang, model):
    return f"{lang}-{model}"


## Connect to ES

In [ ]:

es = Elasticsearch([ES_HOST])
print(es.info()["version"])


## Index Creation — Instructor Format

In [ ]:

def create_index(lang, model):
    idx = index_name(lang, model)
    if es.indices.exists(index=idx):
        es.indices.delete(index=idx)

    analyzer = {
        "analysis": {
            "analyzer": {
                "my_analyzer": {
                    "tokenizer": "whitespace",
                    "filter": ["lowercase", "porter_stem"]
                }
            }
        }
    }

    if model == "tfidf":
        similarity = {
            "similarity": {
                "default": {
                    "type": "scripted",
                    "script": {
                        "source": '''
                            double tf = doc.freq;
                            double idf = Math.log((field.docCount + 1.0)/(term.docFreq + 1.0)) + 1.0;
                            return query.boost * tf * idf;
                        '''
                    }
                }
            }
        }

    elif model == "dirichlet":
        similarity = {
            "similarity": {
                "default": {
                    "type": "LMDirichlet"
                }
            }
        }

    else:  # bm25 default
        similarity = {}

    body = {
        "settings": {
            **analyzer,
            **({"index": similarity} if model in ["tfidf", "dirichlet"] else {}),
            "number_of_shards": 1
        },
        "mappings": {
            "properties": {
                "qid": {"type": "keyword"},
                "title": {"type": "text", "analyzer": "my_analyzer"},
                "body": {"type": "text", "analyzer": "my_analyzer"},
                "answer": {"type": "text", "analyzer": "my_analyzer"}
            }
        }
    }

    es.indices.create(index=idx, body=body)
    print("Created:", idx)

for l in LANGS:
    for m in ["tfidf","bm25","dirichlet"]:
        create_index(l, m)


## Parse qid2all (Tab-Separated Expected)

In [ ]:

def parse_docs(path):
    with open(path, encoding="utf-8", errors="ignore") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 4:
                continue
            yield {
                "qid": parts[0],
                "title": parts[1],
                "body": parts[2],
                "answer": "\t".join(parts[3:])
            }


## Bulk Indexing

In [ ]:

def bulk_index(lang, max_docs=5000):
    models = ["tfidf","bm25","dirichlet"]
    docs = []
    for i, d in enumerate(parse_docs(QID2ALL[lang])):
        docs.append(d)
        if max_docs and i >= max_docs:
            break

    for m in models:
        idx = index_name(lang, m)
        actions = ({
            "_index": idx,
            "_id": d["qid"],
            "_source": d
        } for d in docs)

        for _ in streaming_bulk(es, actions, chunk_size=1000):
            pass

        es.indices.refresh(index=idx)
        print("Indexed:", idx)

for l in LANGS:
    bulk_index(l, max_docs=5000)


## Parse cosidf

In [ ]:

def parse_cosidf(path):
    out = {}
    with open(path, encoding="utf-8", errors="ignore") as f:
        for line in f:
            parts = re.split(r"\s+", line.strip())
            if len(parts) < 3:
                continue
            q1,q2,label = parts[0], parts[1], int(float(parts[2]))
            out.setdefault(q1, []).append((q2,label))
    return out

cosidf = {l: parse_cosidf(COSIDF[l]) for l in LANGS}


## Ranking Request (NDCG@10)

In [ ]:

def make_ratings(index, judged):
    return [{"_index": index, "_id": q2, "rating": int(lbl)} for q2,lbl in judged]

def ranking_body(qid, title, ratings):
    return {
        "requests": [{
            "id": qid,
            "request": {
                "query": {
                    "bool": {
                        "must_not": {"match": {"_id": qid}},
                        "should": [
                            {"match": {"title": {"query": title, "boost": 3.0, "analyzer": "my_analyzer"}}},
                            {"match": {"body": {"query": title, "boost": 0.5, "analyzer": "my_analyzer"}}},
                            {"match": {"answer": {"query": title, "boost": 0.5, "analyzer": "my_analyzer"}}}
                        ]
                    }
                }
            },
            "ratings": ratings
        }],
        "metric": {"dcg": {"k": 10, "normalize": True}}
    }


## Evaluate All 9 Indices

In [ ]:

results = []

for l in LANGS:
    for m in ["tfidf","bm25","dirichlet"]:
        idx = index_name(l,m)
        scores = []
        for qid in list(cosidf[l].keys())[:200]:
            try:
                title = es.get(index=idx, id=qid)["_source"]["title"]
            except:
                continue
            ratings = make_ratings(idx, cosidf[l][qid])
            body = ranking_body(qid, title, ratings)
            try:
                res = es.rank_eval(index=idx, body=body)
                scores.append(res["metric_score"])
            except:
                scores.append(0)
        mean_score = sum(scores)/len(scores) if scores else 0
        results.append((idx, mean_score))
        print(idx, "NDCG@10 =", round(mean_score,6))


## Save Final Report

In [ ]:

with open("hw2_ndcg_report.txt","w") as f:
    for idx,score in results:
        f.write(f"{idx}\t{score}\n")
print("Report saved.")
